In [1]:
import duckdb
import pandas as pd

In [2]:
def load(slice_name):
    return duckdb.connect().sql(f"""
        SELECT * REPLACE (
            coalesce(postcode_clean, '<NONE>') AS postcode_clean,
            coalesce(street_clean,   '<NONE>') AS street_clean
        )
        FROM 'data/raw/{slice_name}_linkage_input.parquet'
    """).df()

def street_key_expr(col: str = "street_clean") -> str:
    """
    Normalise a cleaned street name for grouping: collapses spacing
    variants and a trailing plural/possessive 's'. Does not attempt
    to fix missing/extra words - only exact respellings of the same
    tokens.
    """
    return f"""
        regexp_replace(
            regexp_replace({col}, '[^A-Z0-9]', '', 'g'),  -- drop spaces/apostrophes
            'S$', ''                                       -- drop one trailing S
        )
    """

def ex_key_match(slice_name):
    df = load(slice_name)
    street_key = street_key_expr("street_clean")
    return duckdb.connect().sql(f"""
        SELECT *,
               dense_rank() OVER (
                   ORDER BY postcode_clean, paon_clean, unit_key, street_key
               ) AS property_id
        FROM (
            SELECT *, {street_key} AS street_key
            FROM df
        )
    """).df()

In [3]:
res_bris = ex_key_match("bristol")
res_pow = ex_key_match("powys")

display(res_bris[["property_id", "postcode_clean", "paon_clean", "street_clean", "street_key"]].head())
display(res_pow[["property_id", "postcode_clean", "paon_clean", "street_clean", "street_key"]].head())

,property_id,postcode_clean,paon_clean,street_clean,street_key
0,1,<NONE>,1-2,GLOUCESTER STREET,GLOUCESTERSTREET
1,1,<NONE>,1-2,GLOUCESTER STREET,GLOUCESTERSTREET
2,2,<NONE>,10,BEACONSFIELD ROAD,BEACONSFIELDROAD
3,3,<NONE>,101,EAST STREET,EASTSTREET
4,4,<NONE>,106,CITY ROAD,CITYROAD


,property_id,postcode_clean,paon_clean,street_clean,street_key
0,1,<NONE>,1,BLUEBELL COTTAGES,BLUEBELLCOTTAGE
1,1,<NONE>,1,BLUEBELL COTTAGES,BLUEBELLCOTTAGE
2,2,<NONE>,10,LLEDAN TERRACE,LLEDANTERRACE
3,3,<NONE>,17,LON BRYNFA,LONBRYNFA
4,4,<NONE>,2,ARDDLEEN,ARDDLEEN


In [4]:
count_bris = res_bris.groupby("property_id").size()
count_pow  = res_pow.groupby("property_id").size()

summary = pd.DataFrame({
    "bristol": {
        "n_properties": count_bris.size,
        "n_sales": count_bris.sum(),
        "biggest_group": count_bris.max(),
    },
    "powys": {
        "n_properties": count_pow.size,
        "n_sales": count_pow.sum(),
        "biggest_group": count_pow.max(),
    },
})
summary

,bristol,powys
n_properties,112792,30854
n_sales,220458,53119
biggest_group,10,8


In [5]:
con = duckdb.connect()
con.sql("""
    SELECT postcode_clean, paon_clean, unit_key, street_key,
           count(DISTINCT street_clean) AS n_variants,
           array_agg(DISTINCT street_clean) AS variants
    FROM res_bris
    GROUP BY postcode_clean, paon_clean, unit_key, street_key
    HAVING count(DISTINCT street_clean) > 1
""").show(max_rows=20)

con.sql("""
    SELECT postcode_clean, paon_clean, unit_key, street_key,
           count(DISTINCT street_clean) AS n_variants,
           array_agg(DISTINCT street_clean) AS variants
    FROM res_pow
    GROUP BY postcode_clean, paon_clean, unit_key, street_key
    HAVING count(DISTINCT street_clean) > 1
""").show(max_rows=60)

┌────────────────┬─────────────────┬──────────┬───────────────┬────────────┬───────────────────────────────────┐
│ postcode_clean │   paon_clean    │ unit_key │  street_key   │ n_variants │             variants              │
│    varchar     │     varchar     │ varchar  │    varchar    │   int64    │             varchar[]             │
├────────────────┼─────────────────┼──────────┼───────────────┼────────────┼───────────────────────────────────┤
│ BS9 1PQ        │ SAMBOURNE       │ 3        │ SEAWALLSROAD  │          2 │ [SEAWALLS ROAD, SEA WALLS ROAD]   │
│ BS14 0HL       │ 16              │ <NONE>   │ GREENACREROAD │          2 │ [GREENACRE ROAD, GREEN ACRE ROAD] │
│ BS14 0HL       │ 1               │ <NONE>   │ GREENACREROAD │          2 │ [GREEN ACRE ROAD, GREENACRE ROAD] │
│ BS9 1PG        │ SEAWALLS        │ 8        │ SEAWALLSROAD  │          2 │ [SEA WALLS ROAD, SEAWALLS ROAD]   │
│ BS9 1PG        │ SAMBOURNE LODGE │ <NONE>   │ SEAWALLSROAD  │          2 │ [SEAWALLS ROAD, SEA